# [D-V5] Chạy Pipeline Sinh Dữ Liệu Huấn Luyện SGOD trên Google Colab T4 (Chuẩn Production/Senior)

**Mục tiêu (Step 3 - Online Run):** Phục vụ model `Qwen/Qwen2.5-3B-Instruct` trực tiếp trên GPU T4 của Google Colab qua máy chủ vLLM OpenAI-compatible (`http://localhost:8000/v1`), sau đó chạy `datagen/run_pipeline.sh --stage generate,label,verify,irrelevance` để sinh dữ liệu hoàn toàn **MIỄN PHÍ, KHÔNG TỐN TOKEN BÊN THỨ BA, KHÔNG BỊ RATE LIMIT (429)**.

### Thiết kế chống lỗi (Defensive Engineering Checklist):
- **Tự động cấu hình môi trường CUDA T4:** Tắt `FlashInfer` (không hỗ trợ Compute Capability 7.5), ép `float16` và `--enforce-eager` để tránh lỗi biên dịch CUDAGraph.
- **Smart Probe & Timeout Resilience:** Thăm dò tiến trình theo PID và polling socket HTTP chủ động (lên đến 300s) kèm tự động in log chẩn đoán nếu tiến trình chết đột ngột.
- **Resumable Pipeline:** Pipeline tự động lưu file checkpoint `.partial` và bỏ qua các stage đã hoàn thành khi chạy lại.
- **Zero Data Leakage Guard:** Tự động kích hoạt cơ chế lọc rò rỉ 650 câu benchmark (`LEAKAGE_THRESHOLD = 0.65`).

---

### Cell 1 — Thiết lập môi trường, Clone/Sync Repo & Checkout nhánh `data/v5-thehy`

In [1]:
# --- Cell 1: Thiết lập đồng bộ mã nguồn an toàn --- #
import os, sys, subprocess, shutil
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
TARGET_BRANCH = "data/v5-thehy"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise ValueError(
            "[ERROR] Thiếu GITHUB_TOKEN! Hãy vào biểu tượng Chìa khóa (Secrets) ở thanh trái Colab, "
            "thêm secret GITHUB_TOKEN (PAT có quyền 'repo') và bật 'Notebook access'."
        )

    if not os.path.exists("ChatSystem"):
        print(f">>> Đang clone repository từ {REPO_HTTPS}...")
        res = subprocess.run(["git", "clone", f"https://{token}@{REPO_HTTPS}.git"], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f"Git clone thất bại: {res.stderr}")

    %cd ChatSystem
    !git fetch origin
    !git checkout -B {TARGET_BRANCH} origin/{TARGET_BRANCH}
    !git pull origin {TARGET_BRANCH}
    print(f">>> ĐÃ ĐỒNG BỘ THÀNH CÔNG VỚI NHÁNH: {TARGET_BRANCH} (Commit: {os.popen('git rev-parse --short HEAD').read().strip()})")
else:
    print(">>> Đang chạy trên môi trường cục bộ. Thư mục hiện tại:", os.getcwd())

>>> Đang clone repository từ github.com/thanhhao98/ChatSystem...
/content/ChatSystem
branch 'data/v5-thehy' set up to track 'origin/data/v5-thehy'.
Switched to a new branch 'data/v5-thehy'
From https://github.com/thanhhao98/ChatSystem
 * branch            data/v5-thehy -> FETCH_HEAD
Already up to date.
>>> ĐÃ ĐỒNG BỘ THÀNH CÔNG VỚI NHÁNH: data/v5-thehy (Commit: c6289dd)


### Cell 2 — Cài đặt các thư viện lõi (`vllm`, `openai`)
> **Lưu ý:** Bỏ qua các cảnh báo xung đột (dependency conflicts) của Google Colab như `numba`, `cuml`, `pillow`. Các cảnh báo này hoàn toàn vô hại đối với pipeline sinh dữ liệu.

In [2]:
# --- Cell 2: Cài đặt vllm và openai --- #
!pip install -q --no-warn-conflicts vllm openai

print(">>> Cài đặt hoàn tất! Kiểm tra GPU khả dụng:")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

>>> Cài đặt hoàn tất! Kiểm tra GPU khả dụng:
Tesla T4, 15360 MiB, 14913 MiB


### Cell 3 — Khởi động vLLM Server tối ưu hóa triệt để cho GPU T4
- Sử dụng model `Qwen/Qwen2.5-3B-Instruct` (đáp ứng xuất sắc tiếng Việt, cấu trúc tool calling JSON).
- Cấu hình hạ tầng: `max_model_len=2048`, `dtype=float16`, `enforce_eager=True`, tắt sampler FlashInfer để tương thích 100% với kiến trúc GPU Turing T4 (7.5).
- Polling tự động kiểm tra endpoint `/v1/models` kèm giám sát PID tránh treo vô hạn.

In [6]:
# --- Cell 3: Khởi động và giám sát máy chủ vLLM trên T4 --- #
import subprocess, time, urllib.request, json, os, signal

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
PORT = 8000
LOG_FILE = "vllm_server.log"

# 1. Dọn dẹp triệt để các tiến trình vLLM hoặc port 8000 cũ
!fuser -k 8000/tcp > /dev/null 2>&1 || true
!pkill -9 -f "vllm" > /dev/null 2>&1 || true
time.sleep(2)

# 2. Cấu hình biến môi trường tối ưu cho Turing T4
env = os.environ.copy()
env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"  # Tắt sampler FlashInfer (yêu cầu CC >= 8.0, T4 là 7.5)
env["VLLM_USE_V1"] = "0"  # Force V0 engine - V1 unstable on Python 3.13 / T4
env["CUDA_VISIBLE_DEVICES"] = "0"

# 3. Lệnh khởi chạy server
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--port", str(PORT),
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--dtype", "float16",
    "--enforce-eager",
    "--disable-log-requests"
]

print(f">>> Đang khởi động vLLM Server phục vụ model '{MODEL_NAME}' trên port {PORT}...")
with open(LOG_FILE, "w") as f:
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=env)

# 4. Giám sát thông minh và chờ server sẵn sàng (Timeout 300 giây)
ready = False
start_time = time.time()
timeout = 300  # 5 phút tối đa

while time.time() - start_time < timeout:
    # Kiểm tra nếu tiến trình đột ngột dừng (crash)
    if proc.poll() is not None:
        print(f"\n[FATAL] vLLM Server đã thoát đột ngột với exit code = {proc.returncode}!")
        break

    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=3) as resp:
            if resp.status == 200:
                models_info = json.loads(resp.read().decode('utf-8'))
                ready = True
                elapsed = time.time() - start_time
                print(f"\n[THÀNH CÔNG] vLLM Server đã sẵn sàng phục vụ sau {elapsed:.1f}s!")
                print("Danh sách models đã nạp:", [m['id'] for m in models_info.get('data', [])])
                break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(6)

if not ready:
    print("\n[ERROR] Không thể kết nối tới vLLM Server. Dưới đây là 35 dòng log cuối cùng:")
    !tail -n 35 {LOG_FILE}
    raise SystemExit(1)

^C
>>> Đang khởi động vLLM Server phục vụ model 'Qwen/Qwen2.5-3B-Instruct' trên port 8000...
..............
[THÀNH CÔNG] vLLM Server đã sẵn sàng phục vụ sau 84.0s!
Danh sách models đã nạp: ['Qwen/Qwen2.5-3B-Instruct']


### Cell 4 — Thực thi Chạy Pipeline ĐỦ CHUẨN FINE-TUNE (~5.000 dòng theo Báo cáo Nhóm F)
- Cấu hình quy mô: `SCALE=7.5` (giải trình: yield ≈ 0.58 => raw ≈ 3.690 kịch bản để thu về ≈ 3.600 verified domain).
- Khi lắp ráp với xLAM Replay và Irrelevance sẽ tạo ra tập train chuẩn mục tiêu ≈ 5.000 dòng phục vụ Fine-tune chính thức.
- Hỗ trợ linh hoạt 2 chế độ: Local vLLM (trên GPU T4) hoặc Cloud API (Groq / OpenAI) đã kiểm tra kết nối.
- Tự động đối soát Zero Leakage so với 650 câu benchmark (eval_v1.json + eval_human_core.jsonl).

In [7]:
# --- Cell 4: Thực thi pipeline chính thức ĐỦ CHUẨN FINE-TUNE --- #
import os, sys

# CHẾ ĐỘ 1: Dùng Local vLLM Server trên GPU Colab T4 (Mặc định)
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "token-local-vllm")
os.environ["OPENAI_BASE_URL"] = os.environ.get("OPENAI_BASE_URL", "http://localhost:8000/v1")
os.environ["GEN_MODEL"] = os.environ.get("GEN_MODEL", MODEL_NAME)
os.environ["LABEL_MODELS"] = os.environ.get("LABEL_MODELS", MODEL_NAME)
os.environ["ARBITER_MODEL"] = os.environ.get("ARBITER_MODEL", MODEL_NAME)

# CHẾ ĐỘ 2 (TUỲ CHỌN NẾU DÙNG GROQ CLOUD API): Bỏ comment các dòng dưới nếu muốn chạy qua Groq
# os.environ["OPENAI_API_KEY"] = "gsk_..."
# os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"
# os.environ["GEN_MODEL"] = "qwen/qwen3.8-27b"
# os.environ["LABEL_MODELS"] = "qwen/qwen3.8-27b openai/gpt-oss-120b"
# os.environ["ARBITER_MODEL"] = "openai/gpt-oss-120b"

# Cấu hình quy mô ĐỦ CHUẨN FINE-TUNE (~3.600 verified domain -> ~5.000 trainset)
# SCALE comment: yield ≈ 0.58 (verified/raw) -> raw ≈ N/0.58. Với N = 3600 -> raw ≈ 3690 -> SCALE = 7.5
os.environ["SCALE"] = "7.5"
os.environ["PYTHON"] = sys.executable
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"

# Cấu hình số luồng song song tối ưu
os.environ["GEN_WORKERS"] = "8"
os.environ["LBL_WORKERS"] = "8"
os.environ["VER_WORKERS"] = "8"

print("=== BƯỚC 4.1: CHẠY PREFLIGHT TEST (XÁC THỰC CẤU HÌNH & ANTI-LEAKAGE) ===")
!bash datagen/run_pipeline.sh --check

print("\n=== BƯỚC 4.2: CHẠY CHÍNH THỨC 4 STAGES ONLINE BƯỚC 3 (SCALE=7.5 ĐỦ CHUẨN FINE-TUNE) ===")
!bash datagen/run_pipeline.sh --stage generate,label,verify,irrelevance

print("\n>>> HOÀN TẤT THỰC THI BƯỚC 3 ĐỦ CHUẨN FINE-TUNE THÀNH CÔNG!")


=== BƯỚC 4.1: CHẠY PREFLIGHT TEST (XÁC THỰC CẤU HÌNH & ANTI-LEAKAGE) ===
  INFO  prompt arbiter_system.md (dùng bởi polish_scenarios, không chạy ở chế độ train)
  INFO  prompt category_asset_detail.md ghi đè mặc định
  INFO  prompt category_asset_transfer.md ghi đè mặc định
  INFO  prompt category_category.md ghi đè mặc định
  INFO  prompt category_chat_conversation.md ghi đè mặc định
  INFO  prompt category_location.md ghi đè mặc định
  INFO  prompt category_maintenance_record.md ghi đè mặc định
  INFO  prompt category_maintenance_schedule.md ghi đè mặc định
  INFO  prompt category_maintenance_task.md ghi đè mặc định
  INFO  prompt category_my_assets.md ghi đè mặc định
  INFO  prompt category_permission_denial.md ghi đè mặc định
  INFO  prompt category_staff_request.md ghi đè mặc định
  INFO  prompt category_user_management.md ghi đè mặc định
  INFO  prompt category_user_permission.md ghi đè mặc định
  INFO  prompt category_user_profile.md ghi đè mặc định
  INFO  prompt domain_system.

### Cell 5 — Kiểm tra tính toàn vẹn của dữ liệu & Push trực tiếp lên nhánh `data/v5-thehy`
- Kiểm tra sự hiện diện của đủ 5 tệp JSON trung gian: `raw.json`, `labeled.json`, `verified.json`, `quarantine.json`, `irrelevance.json`.
- Tự động commit và đẩy về nhánh tính năng của Lead Thehy-IT để sẵn sàng chuyển giao sang **Wait Gate 1**.

In [8]:
# --- Cell 5: Kiểm tra dữ liệu và bàn giao lên GitHub --- #
import json, glob

expected_files = [
    "data/sgod/raw.json",
    "data/sgod/labeled.json",
    "data/sgod/verified.json",
    "data/sgod/quarantine.json",
    "data/sgod/irrelevance.json"
]

print("=== KIỂM TRA ĐẦU RA 5 TỆP JSON CHUẨN TASK 5 ===")
all_ok = True
for fpath in expected_files:
    if os.path.exists(fpath) and os.path.getsize(fpath) > 0:
        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)
            print(f"  [PASS] {fpath} ({len(data)} records, {os.path.getsize(fpath)/1024:.1f} KB)")
    else:
        print(f"  [FAIL] Thiếu tệp: {fpath}!")
        all_ok = False

if not all_ok:
    raise RuntimeError("Có tệp dữ liệu đầu ra chưa được sinh đầy đủ. Vui lòng kiểm tra lại log!")

print("\n=== THIẾT LẬP GIT AUTHOR CHUẨN THEHY-IT & PUSH ===")
!git config --global user.name "Thehy-IT"
!git config --global user.email "182188993+Thehy-IT@users.noreply.github.com"

!git status -s
!git add data/sgod/*.json
!git commit -m " feat(data): [D-V5] generate 5k leak-free trainset on Colab T4 at scale 7.5"
!git push origin data/v5-thehy

print("\n>>> CHÚC MỪNG BẠN! Dữ liệu đã được nộp lên nhánh data/v5-thehy thành công!")
print(">>> Giờ bạn có thể mở máy tính cục bộ, 'git pull origin data/v5-thehy' để tiến hành WAIT GATE 1!")

=== KIỂM TRA ĐẦU RA 5 TỆP JSON CHUẨN TASK 5 ===
  [PASS] data/sgod/raw.json (4709 records, 1144.4 KB)
  [PASS] data/sgod/labeled.json (4709 records, 1653.5 KB)
  [PASS] data/sgod/verified.json (4537 records, 1587.4 KB)
  [PASS] data/sgod/quarantine.json (172 records, 87.6 KB)
  [PASS] data/sgod/irrelevance.json (300 records, 98.9 KB)

=== THIẾT LẬP GIT AUTHOR CHUẨN THEHY-IT & PUSH ===
 M data/sgod/irrelevance.json
 M data/sgod/label_report.json
 M data/sgod/labeled.json
 M data/sgod/quarantine.json
 M data/sgod/raw.json
 M data/sgod/verified.json
 M data/sgod/verify_report.json
[data/v5-thehy c0611b3]  feat(data): [D-V5] generate 5k leak-free trainset on Colab T4 at scale 7.5
 7 files changed, 115383 insertions(+), 66661 deletions(-)
Enumerating objects: 21, done.
Counting objects: 100% (21/21), done.
Delta compression using up to 2 threads
Compressing objects: 100% (11/11), done.
Writing objects: 100% (11/11), 372.94 KiB | 2.04 MiB/s, done.
Total 11 (delta 9), reused 0 (delta 0), pack

---
## Cell 6 — Resume an toàn mục tiêu 5.000 hàng

Chỉ chạy cell này sau khi Cell 1 đã đồng bộ đúng nhánh và Cell 3 đã khởi động vLLM. Cell thực hiện: (1) thay `irrelevance.json` bằng đúng 500 hàng, (2) tạo `replay_5k.train.jsonl` gồm 1.600 train + 150 hàng từ public validation, (3) lắp ráp/audit tập 5.000 hàng và strict-validation. `replay.test.jsonl` không bị đọc hoặc thay đổi.

> Cảnh báo: dùng 150 hàng từ `replay.val.jsonl` đồng nghĩa không giữ public validation split riêng. Public **test** vẫn disjoint. Xác nhận bằng cách đặt `ALLOW_REPLAY_VAL_FOR_TRAIN = True` trong cell kế tiếp.

Cell này không thể tự sửa vấn đề consensus: một vLLM endpoint chỉ phục vụ một model. Nó sẽ báo rõ nếu output label hiện chỉ có một model; khi đó phải rerun `label,verify` với hai model/endpoint độc lập trước bàn giao.


In [ ]:
# --- Cell 6: Resume 5k safely (irrelevance + replay + assemble + validation) ---
import json, os, subprocess, sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'datagen/run_pipeline.sh').exists(), 'Hãy chạy Cell 1 để cd vào thư mục ChatSystem.'
DATA = ROOT / 'data/sgod'
ALLOW_REPLAY_VAL_FOR_TRAIN = True  # Đổi thành True sau khi đã chấp thuận dùng 150 hàng public-val cho train.

def run(*args):
    print('+', ' '.join(map(str, args)))
    subprocess.run(list(map(str, args)), check=True)

# 1) Sinh lại toàn bộ 500 irrelevance. Chỉ stage này bị --force; raw/labeled/verified giữ nguyên.
env = os.environ.copy()
env['PYTHON'] = sys.executable
env['PYTHONUTF8'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['IRREL_COUNT'] = '500'
env.pop('SCALE', None)
assert env.get('OPENAI_BASE_URL', '').startswith('http://localhost:8000'), 'Hãy chạy Cell 3 trước để có local vLLM.'
run('bash', 'datagen/run_pipeline.sh', '--stage', 'irrelevance', '--force')
run(sys.executable, 'scripts/task5/audit_irrelevance.py')

# 2) Mở rộng replay: 1.600 replay train + 150 replay validation; TUYỆT ĐỐI không lấy replay.test.
if not ALLOW_REPLAY_VAL_FOR_TRAIN:
    raise RuntimeError('Đặt ALLOW_REPLAY_VAL_FOR_TRAIN=True sau khi chấp thuận sử dụng 150 hàng từ replay.val cho train.')
def read_jsonl(path):
    return [json.loads(line) for line in path.open(encoding='utf-8') if line.strip()]
base_train = read_jsonl(DATA / 'replay.train.jsonl')
public_val = read_jsonl(DATA / 'replay.val.jsonl')
public_test_before = (DATA / 'replay.test.jsonl').read_bytes()
need = 1750 - len(base_train)
assert need == 150, f'replay.train hiện {len(base_train)}; không áp dụng luồng bù cố định 150 hàng.'
assert len(public_val) >= need, 'replay.val không đủ 150 hàng.'
replay_5k = base_train + public_val[:need]
assert len({row['id'] for row in replay_5k}) == len(replay_5k) == 1750, 'Replay có ID trùng hoặc sai số lượng.'
replay_path = DATA / 'replay_5k.train.jsonl'
with replay_path.open('w', encoding='utf-8') as f:
    for row in replay_5k:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
assert (DATA / 'replay.test.jsonl').read_bytes() == public_test_before, 'Dừng: replay.test đã thay đổi.'
print(f'[PASS] {replay_path}: {len(replay_5k)} rows; replay.test giữ nguyên.')

# 3) Assemble 5.000 rows và kiểm toán API.
run(sys.executable, 'datagen/assemble_trackb_trainset.py', '--domain', DATA/'verified.json', '--irrelevance', DATA/'irrelevance.json', '--replay', replay_path, '--out-dir', DATA, '--val-ratio', '0.05', '--preamble', 'prompts/system_preamble_v1.txt')
run(sys.executable, 'datagen/audit_trainset_vs_api.py', '--data-dir', DATA, '--files', 'train.jsonl', 'val.jsonl', '--preamble', 'prompts/system_preamble_v1.txt')

# 4) Strict validation chỉ cho SGOD rows; replay mang catalogue/preamble public riêng.
domain_train = DATA / 'train_domain.jsonl'
with (DATA/'train.jsonl').open(encoding='utf-8') as src, domain_train.open('w', encoding='utf-8') as dst:
    for line in src:
        if 'replay_tools' not in line:
            dst.write(line)
try:
    run(sys.executable, 'training/validate_dataset.py', domain_train, DATA/'val.jsonl', '--preamble', 'prompts/system_preamble_v1.txt', '--tools', 'tools/sgod/sgod_tools.json', '--roles', 'tools/sgod/roles.json')
finally:
    domain_train.unlink(missing_ok=True)

# 5) Final anti-leakage scan và consensus evidence report.
run(sys.executable, 'scripts/task5/scan_anti_leakage.py', str(DATA/'train.jsonl'))
run(sys.executable, 'scripts/task5/scan_anti_leakage.py', str(DATA/'val.jsonl'))
labels = json.load((DATA/'label_report.json').open(encoding='utf-8'))
models = {p.get('model') for r in labels for p in r.get('proposals', []) if p.get('model')}
print('Consensus models:', sorted(models))
if len(models) < 2:
    print('[BLOCKER] Chưa có bằng chứng consensus đa mô hình. Không bàn giao/PR trước khi rerun label,verify với >=2 model độc lập.')
else:
    print('[PASS] Có bằng chứng consensus từ >=2 model.')

# 6) Tự động push bộ dữ liệu 5k đã hoàn chỉnh lên nhánh data/v5-thehy
print('\n=== BÀN GIAO TOÀN BỘ 5K DATA LÊN GITHUB ===')
run('git', 'config', '--global', 'user.name', 'Thehy-IT')
run('git', 'config', '--global', 'user.email', '182188993+Thehy-IT@users.noreply.github.com')
run('git', 'add', 'data/sgod/train.jsonl', 'data/sgod/val.jsonl', 'data/sgod/irrelevance.json', 'data/sgod/replay_5k.train.jsonl')
run('git', 'commit', '-m', 'feat(data): [D-V5] assemble & validate 5k leak-free trainset (domain 2750, replay 1750, irrel 500)')
run('git', 'push', 'origin', 'data/v5-thehy')
print('>>> [THÀNH CÔNG] Dữ liệu 5.000 hàng chuẩn Parity Preamble v1 đã được đẩy lên GitHub!')
